# C-GCN Demo

## Introduction

The goal of this notebook is to showcase how to use and test the pre-trained modified C-GCN model.

The notebook is structured as follows:
1. Setup (Dependencies, imports and function definitions)
2. Loading the model
3. Demonstration of executing the model on the test data
4. Model inference, i.e. demonstration of model execution on user provided data


### ATTENTION: If using Google Colab, please upload the src_gcn.zip and data.zip files to the session storage and make sure that a CUDA-enabled runtime is being used! This notebook was tested on the T4 runtime.


## 1. Setup (Dependencies, Imports and Function Definitions)

### If using Google Colab, please upload the src_cgn.zip and data.zip files to the session storage!

In [ ]:
%pip install torchao torchtune

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 117.0/117.0 kB 9.6 MB/s eta 0:00:00
  Preparing metadata (setup.py) ... done
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 5.7/5.7 MB 110.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 810.3/810.3 kB 58.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 75.4/75.4 kB 8.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 485.4/485.4 kB 39.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 79.5/79.5 kB 8.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.2/1.2 MB 71.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 116.3/116.3 kB 13.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 3.6/3.6 MB 107.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 143.5/143.5 kB 16.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 2.3/2.3 MB 94.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 194.8/194.8 kB 20.1 MB/s eta 0:00:00
  Crea

In [ ]:
!if [ ! -d src_cgcn ]; then unzip src_cgcn.zip; fi
!if [ ! -d data ]; then unzip data.zip; fi

Archive:  src_cgcn.zip
   creating: src_cgcn/
   creating: src_cgcn/data/
  inflating: src_cgcn/data/loader.py  
   creating: src_cgcn/data/__pycache__/
  inflating: src_cgcn/data/__pycache__/loader.cpython-38.pyc  
   creating: src_cgcn/dataset/
   creating: src_cgcn/dataset/vocab/
  inflating: src_cgcn/dataset/vocab/embedding.npy  
  inflating: src_cgcn/dataset/vocab/vocab.pkl  
  inflating: src_cgcn/download.sh    
  inflating: src_cgcn/eval.py        
   creating: src_cgcn/fig/
  inflating: src_cgcn/fig/architecture.png  
  inflating: src_cgcn/fig/examples.png  
  inflating: src_cgcn/LICENSE        
   creating: src_cgcn/model/
  inflating: src_cgcn/model/gcn.py   
  inflating: src_cgcn/model/trainer.py  
  inflating: src_cgcn/model/tree.py  
   creating: src_cgcn/model/__pycache__/
  inflating: src_cgcn/model/__pycache__/gcn.cpython-38.pyc  
  inflating: src_cgcn/model/__pycache__/trainer.cpython-38.pyc  
  inflating: src_cgcn/model/__pycache__/tree.cpython-38.pyc  
  inflating: s

In [ ]:
%cd src_cgcn

/content/src_cgcn


In [ ]:
# imports

import random
import argparse

from tqdm import tqdm
import torch

from data.loader import DataLoader, DataLoaderPredict
from model.trainer import GCNTrainer
from utils import torch_utils, scorer, constant, helper
from utils.vocab import Vocab


In [ ]:
def load_model(model_dir, model="best_model.pt", seed=1234):
    # check cuda set and cuda seed
    if torch.cuda.is_available():
        torch.cuda.manual_seed(seed)

    # set the seeds
    torch.manual_seed(seed)
    random.seed(seed)

    # load opt
    model_file = f"{model_dir}/{model}"
    print(f"Loading model from {model_file}")
    opt = torch_utils.load_config(model_file)
    trainer = GCNTrainer(opt)
    trainer.load(model_file)

    # load vocab
    vocab_file = f"{model_dir}/vocab.pkl"
    vocab = Vocab(vocab_file, load=True)
    assert opt['vocab_size'] == vocab.size, "Vocab size must match that in the saved model."

    return trainer, vocab, opt

In [ ]:
def evaluate_model(trainer, vocab, data_dir="../data/retacred", opt=None, scorer=None):
    assert opt is not None, "opt must not be None."
    assert scorer is not None, "scorer must not be None."

    # load data
    data_file = f"{data_dir}/test.json"
    print(f"Loading data from {data_file} with batch size {opt['batch_size']}...")
    batch = DataLoader(data_file, opt['batch_size'], opt, vocab, evaluation=True)

    label2id = constant.LABEL_TO_ID
    id2label = dict([(v,k) for k,v in label2id.items()])

    predictions = []
    all_probs = []
    batch_iter = tqdm(batch)
    for i, b in enumerate(batch_iter):
        preds, probs, _ = trainer.predict(b)
        predictions += preds
        all_probs += probs

    predictions = [id2label[p] for p in predictions]
    _, details = scorer.score(batch.gold(), predictions, verbose=False)

    # print("Detailed evaluation:")
    # print("\n".join([f"{k}: {v}" for k, v in details.items()]))
    print("Evaluation ended.")

In [ ]:
def predict_relation(input, opt, trainer, vocab):
    print(f"Input sentence: {' '.join(input[0]['token'])}")
    print(f"Subject: {' '.join(input[0]['token'][input[0]['subj_start']:input[0]['subj_end']+1])} ({input[0]['subj_type']})")
    print(f"Object: {' '.join(input[0]['token'][input[0]['obj_start']:input[0]['obj_end']+1])} ({input[0]['obj_type']})")

    batch = DataLoaderPredict(input, opt, vocab)

    # helper.print_config(opt)
    label2id = constant.LABEL_TO_ID
    id2label = dict([(v,k) for k,v in label2id.items()])

    predictions = []
    all_probs = []
    for i, b in enumerate(batch):
        preds, probs, _ = trainer.predict(batch=b, test=True)
        predictions += preds
        all_probs += probs

    predictions = [id2label[p] for p in predictions]
    print(f"Predicted relation: {predictions[0]}")

    return predictions[0]

## 2. Loading the model

In [ ]:
trainer, vocab, opt = load_model(model_dir="saved_models/400", model="best_model.pt")

Loading model from saved_models/400/best_model.pt
Finetune all embeddings.
Vocab size 50115 loaded from file


/content/src_cgcn/model/trainer.py:29: FutureWarning: You are using `torch.load` with `weights_only=False` (the current default value), which uses the default pickle module implicitly. It is possible to construct malicious pickle data which will execute arbitrary code during unpickling (See https://github.com/pytorch/pytorch/blob/main/SECURITY.md#untrusted-models for more details). In a future release, the default value for `weights_only` will be flipped to `True`. This limits the functions that could be executed during unpickling. Arbitrary objects will no longer be allowed to be loaded via this mode unless they are explicitly allowlisted by the user via `torch.serialization.add_safe_globals`. We recommend you start setting `weights_only=True` for any use case where you don't have full control of the loaded file. Please open an issue on GitHub for any issues related to this experimental feature.
  checkpoint = torch.load(filename)


## 3. Model Evaluation

In [ ]:
evaluate_model(trainer=trainer, vocab=vocab, opt=opt, scorer=scorer)

Loading data from ../data/retacred/test.json with batch size 50...
269 batches created for ../data/retacred/test.json


100%|██████████| 269/269 [00:05<00:00, 51.77it/s]


Precision (micro): 78.580%
   Recall (micro): 76.841%
       F1 (micro): 77.701%
Evaluation ended.


In [ ]:
# Evaluation of original GCN on Re-TACRED
trainer_original, vocab_original, opt_original = load_model(model_dir="saved_models/500/", model="best_model.pt")
evaluate_model(trainer=trainer_original, vocab=vocab_original, opt=opt_original, scorer=scorer)

Loading model from saved_models/500//best_model.pt


c:\Users\Alex\Desktop\Text Mining Coursework\COMP61332-re\src_ra_cgcn\utils\torch_utils.py:158: FutureWarning: You are using `torch.load` with `weights_only=False` (the current default value), which uses the default pickle module implicitly. It is possible to construct malicious pickle data which will execute arbitrary code during unpickling (See https://github.com/pytorch/pytorch/blob/main/SECURITY.md#untrusted-models for more details). In a future release, the default value for `weights_only` will be flipped to `True`. This limits the functions that could be executed during unpickling. Arbitrary objects will no longer be allowed to be loaded via this mode unless they are explicitly allowlisted by the user via `torch.serialization.add_safe_globals`. We recommend you start setting `weights_only=True` for any use case where you don't have full control of the loaded file. Please open an issue on GitHub for any issues related to this experimental feature.
  dump = torch.load(filename)


Finetune all embeddings.
Vocab size 50115 loaded from file
Loading data from ../data/retacred/test.json with batch size 50...


c:\Users\Alex\Desktop\Text Mining Coursework\COMP61332-re\src_ra_cgcn\model\trainer.py:29: FutureWarning: You are using `torch.load` with `weights_only=False` (the current default value), which uses the default pickle module implicitly. It is possible to construct malicious pickle data which will execute arbitrary code during unpickling (See https://github.com/pytorch/pytorch/blob/main/SECURITY.md#untrusted-models for more details). In a future release, the default value for `weights_only` will be flipped to `True`. This limits the functions that could be executed during unpickling. Arbitrary objects will no longer be allowed to be loaded via this mode unless they are explicitly allowlisted by the user via `torch.serialization.add_safe_globals`. We recommend you start setting `weights_only=True` for any use case where you don't have full control of the loaded file. Please open an issue on GitHub for any issues related to this experimental feature.
  checkpoint = torch.load(filename)


269 batches created for ../data/retacred/test.json


100%|██████████| 269/269 [00:06<00:00, 40.24it/s]


Precision (micro): 75.224%
   Recall (micro): 77.408%
       F1 (micro): 76.300%
Evaluation ended.


## 4. Model Inference

To use the relation prediction model, the input data needs to be formatted in a specific way.

We require the input data to be a list of dictionaries, where the dictionary contains the following fields:

- `token`: The tokenised sentence

- `subj_start`: the index of the tokenised list where the subject starts

- `subj_end`: the index of the tokenised list where the subject ends

- `obj_start`: the index of the tokenised list where the object starts

- `obj_end`: the index of the tokenised list where the object ends

- `subj_type`: The type of subject, a choice between `ORGANIZATION` and `PERSON`

- `obj_type`: The type of object, a choice between 17 different types: `URL`, `DATE`, `NUMBER`, `RELIGION`, `IDEOLOGY`, `MISC`, `CITY`, `COUNTRY`, `STATE_OR_PROVINCE`, `LOCATION`, `ORGANIZATION`, `PERSON`, `TITLE`, `CRIMINAL_CHARGE`, `CAUSE_OF_DEATH`, `DURATION`, `NATIONALITY`

- `stanford_pos`: The POS tags acquired from Stanford CoreNLP annotations

- `stanford_ner`: The NER tags acquired from Stanford CoreNLP annotations

- `stanford_head`: The dependency tree head indices acquired from Stanford CoreNLP annotations

- `stanford_deprel`: The dependency relation types acquired from Stanford CoreNLP annotations


Please see the examples below to help with the structuring of input data.

If POS tags, NER tags and the dependency tree need to be acquired for a sample sentence, please visit the [CoreNLP Live Online Demo](https://corenlp.run/).

In [ ]:
sample_data = [
    {
        "token": ["He", "has", "served", "as", "a", "policy", "aide", "to", "the", "late", "U.S.", "Senator", "Alan", "Cranston", ",", "as", "National", "Issues", "Director", "for", "the", "2004", "presidential", "campaign", "of", "Congressman", "Dennis", "Kucinich", ",", "as", "a", "co-founder", "of", "Progressive", "Democrats", "of", "America", "and", "as", "a", "member", "of", "the", "international", "policy", "department", "at", "the", "RAND", "Corporation", "think", "tank", "before", "all", "that", "."], 
        "subj_start": 33, 
        "subj_end": 36, 
        "obj_start": 43, 
        "obj_end": 45, 
        "subj_type": "ORGANIZATION", 
        "obj_type": "ORGANIZATION", 
        "stanford_pos": ["PRP", "VBZ", "VBN", "IN", "DT", "NN", "NN", "TO", "DT", "JJ", "NNP", "NNP", "NNP", "NNP", ",", "IN", "NNP", "NNP", "NNP", "IN", "DT", "CD", "JJ", "NN", "IN", "NNP", "NNP", "NNP", ",", "IN", "DT", "NN", "IN", "NNP", "NNPS", "IN", "NNP", "CC", "IN", "DT", "NN", "IN", "DT", "JJ", "NN", "NN", "IN", "DT", "NNP", "NNP", "VB", "NN", "IN", "DT", "DT", "."], 
        "stanford_ner": ["O", "O", "O", "O", "O", "O", "O", "O", "O", "O", "LOCATION", "O", "PERSON", "PERSON", "O", "O", "O", "O", "O", "O", "O", "DATE", "O", "O", "O", "O", "PERSON", "PERSON", "O", "O", "O", "O", "O", "ORGANIZATION", "ORGANIZATION", "ORGANIZATION", "ORGANIZATION", "O", "O", "O", "O", "O", "O", "O", "O", "O", "O", "O", "ORGANIZATION", "ORGANIZATION", "O", "O", "O", "O", "O", "O"], 
        "stanford_head": [3, 3, 0, 7, 7, 7, 3, 14, 14, 14, 14, 14, 14, 3, 3, 19, 19, 19, 3, 24, 24, 24, 24, 19, 28, 28, 28, 24, 3, 32, 32, 3, 35, 35, 32, 37, 35, 32, 41, 41, 32, 46, 46, 46, 46, 41, 50, 50, 50, 46, 41, 51, 54, 52, 54, 3], 
        "stanford_deprel": ["nsubj", "aux", "ROOT", "case", "det", "compound", "nmod", "case", "det", "amod", "compound", "compound", "compound", "nmod", "punct", "case", "compound", "compound", "nmod", "case", "det", "nummod", "amod", "nmod", "case", "compound", "compound", "nmod", "punct", "case", "det", "nmod", "case", "compound", "nmod", "case", "nmod", "cc", "case", "det", "conj", "case", "det", "amod", "compound", "nmod", "case", "det", "compound", "nmod", "acl", "dobj", "case", "nmod", "dep", "punct"]
    }
]

sample_data_2 = [
    {
        "token": ["Her", "companion", "and", "employer", ",", "Laura", "Silsby", ",", "40", ",", "remained", "in", "detention", "."],
        "subj_start": 5,
        "subj_end": 6,
        "obj_start": 8,
        "obj_end": 8,
        "subj_type": "PERSON",
        "obj_type": "NUMBER",
        "stanford_pos": ["PRP$", "NN", "CC", "NN", ",", "NNP", "NNP", ",", "CD", ",", "VBD", "IN", "NN", "."],
        "stanford_ner": ["O", "O", "O", "O", "O", "PERSON", "PERSON", "O", "NUMBER", "O", "O", "O", "O", "O"],
        "stanford_head": [2, 11, 2, 2, 2, 7, 2, 7, 7, 2, 0, 13, 11, 11],
        "stanford_deprel": ["nmod:poss", "nsubj", "cc", "conj", "punct", "compound", "appos", "punct", "amod", "punct", "ROOT", "case", "nmod", "punct"]
  }
]

In [ ]:
predict_relation(input=sample_data, opt=opt, trainer=trainer, vocab=vocab)

Input sentence: He has served as a policy aide to the late U.S. Senator Alan Cranston , as National Issues Director for the 2004 presidential campaign of Congressman Dennis Kucinich , as a co-founder of Progressive Democrats of America and as a member of the international policy department at the RAND Corporation think tank before all that .
Subject: Progressive Democrats of America (ORGANIZATION)
Object: international policy department (ORGANIZATION)
Predicted relation: no_relation


'no_relation'

In [ ]:
predict_relation(input=sample_data_2, opt=opt, trainer=trainer, vocab=vocab)

Input sentence: Her companion and employer , Laura Silsby , 40 , remained in detention .
Subject: Laura Silsby (PERSON)
Object: 40 (NUMBER)
Predicted relation: per:age


'per:age'